## Análisis de la hiperuniformidad con diferentes decorados para N = 23

In [ ]:
#Cargamos las librerías a utilizar para la lectura, análisis y visualización de los datos
using CairoMakie;       #Paquetería para generar gráficos de los datos
using Colors;           #Paquetería para usar RGB en las gráficas de Makie
using DelimitedFiles;   #Paquetería para leer y escribir archivos .csv
using LaTeXStrings;     #Paquetería para emplear formato LaTeX en las gráficas
using LinearAlgebra;    #Paquetería para emplear funciones básicas de álgebra lineal
using Measures;

#Cargamos las funciones necesarias para generar vecindades cuasiperiódicas
PathFunctions = "E:/PostDoc_CAS/CAS_Postdoc_Viz/Tercer_Articulo/Functions/";
#PathFunctions = "Quasiperiodic-Tiles/Global_Structural_Studies/Functions/";
include(PathFunctions * "Quasicrystals.jl");
#Función que calcula el área de un rombo
area(rombo) = abs(0.5*sum((rombo[i][2]+rombo[mod1(i+1,4)][2])*(rombo[i][1]-rombo[mod1(i+1,4)][1]) for i in 1:4));

### Teoría del Factor de Normalización

En la siguiente celda se introduce un factor de normalización al radio dado por:

$$F_{N} = 2 \sqrt{\pi \rho}$$

La idea detrás de este factor es obtener una densidad constante igual a $\frac{1}{4 \pi}$ para todos los sistemas a considerar, como desarrollamos a continuación:

Sea $N_{p}(R)$ el número de puntos que hay dentro de una vecindad circular $V_{R}$ de radio $R$. La densidad de puntos $\rho(R)$ en la vecindad $V_{R}$ es:

$$\rho(R) = \frac{N_{p}(R)}{\pi R^2}$$

Sea $F_{N}$ como se definió anteriormente, sustituyendo la $\rho$ por su aproximación obtenida con la vecindad $V_{R}$ tenemos:

$$F_{N} \approx 2 \sqrt{\pi \rho(R)} = 2 \sqrt{\pi \frac{N_{p}(R)}{\pi R^2}} = 2 \sqrt{\frac{N_{p}(R)}{R^2}} = \frac{2}{R} \sqrt{N_{p}(R)}$$

Escalemos ahora a cualquier radio $r$ por $r^{\ \prime} = (F_{N}) (r) = \left( \frac{2}{R} \sqrt{N_{p}(R)} \right) (r) = 2 \left( \frac{r}{R} \right) \sqrt{N_{p}(R)}$.

La densidad de puntos para una nueva vecindad $V_{r^{\ \prime}}$ está dada por:

$$\rho(r^{\ \prime}) = \frac{N_{p}(r^{\ \prime})}{\pi (r^{\ \prime})^{2}} = \frac{N_{p}(r^{\ \prime})}{\pi \left( 2 \frac{r}{R} \sqrt{N_{p}(R)} \right)^{2}} = \frac{ \frac{N_{p}(r^{\ \prime})}{1} }{ \frac{4 \pi r^2 N_{p}(R)}{R^2} } = \frac{N_{p}(r^{\ \prime}) R^2}{4 \pi r^2 N_{p}(R)} = \frac{1}{4 \pi} \left( \frac{N_{p}(r^{\ \prime})}{r^2} \right) \left( \frac{R^2}{N_{p}(R)} \right)$$

Bajo la hipótesis de que al escalar los radios, el número de puntos dentro de las vecindades circulares no se ven afectados, entonces $N_{p}(r^{\ \prime}) = N_{p}(r)$. Si además consideramos que la densidad del sistema cuasiperiódico es constante (o aproximadamente constante) a lo largo de toda su extensión y en cualquier región lo suficientemente grande de esta, entonces $ \frac{N_{p}(r)}{r^2} \approx \frac{N_{p}(R)}{R^2}$, de donde:

$$\rho(r^{\ \prime}) \approx \frac{1}{4 \pi}$$

Notemos que ahora, la densidad $\rho(r^{\ \prime})$ tiene un valor constante, sin importar de qué sistema cuasiperiódico estemos hablando, contrario al valor de $\rho(R)$ que sí depende del sistema cuasiperiódico en cuestión.

### Decorado: Centroides - Per Area Group

##### Carga de datos y definición de parámetros

In [ ]:
#Generamos lo que será el centro de todas las vecindades que emplearemos
SL = 1e6; #Semilado del cuadrado centrado en el origen donde se generará una vecindad circular del cuasicristal en un sitio arb
APoint = punto_Arbitrario(SL); #Generamos un punto arbitrario dentro del cuadrado centrado en el origen
APoint = [247903.0950577349, -913720.8449383951];

In [ ]:
#Datos de las vecindades generadas al obtener los datos
NSides = 23;    #Simetría rotacional de los sistemas cuasperiódicos a analizar
Radio = 930;    #Radio de las vecindades circulares
Step = 100000;  #Número de pasos a dar desde R = 0 hasta R = Radio para variar el radio de la ventana circular de N(R)

#Densidad de los vértices del decorado para factor de normalización
Rho = 1.2712590235307708;   #Densidad del decorado en vértices, no en centroides
FN = 2*sqrt(π*Rho);         #Factor de normalización para mantener los resultados independientes de la densidad de puntos

#Generamos el intervalo con los valores de la R asociados a los datos de sigma cuadrada (Incluye factor de 2*sqrt(π*Rho)
#necesario para mantener densidad de puntos constantes en decorado, independientemente de la simetría rotacional)
R = (FN*Radio/Step):(FN*Radio/Step):(FN*Radio);

#Cargamos los datos de la sigma cuadrada
PathData = "E:/PostDoc_CAS/CAS_Postdoc_Viz/Tercer_Articulo/Data/gR_NR/Special_Data_Old_NR_N23/";
#PathData = "Quasiperiodic-Tiles/Global Structural Studies/Data/Fig2_SigmaSquare_PerAreaGroup/";
σ2_Centroid_1T = vec(readdlm(PathData * "Torquato_NR_N$(NSides)_Alfa0P0_Area1_SigmaCuadrada.csv"));
σ2_Centroid_2T = vec(readdlm(PathData * "Torquato_NR_N$(NSides)_Alfa0P0_Area2_SigmaCuadrada.csv"));
σ2_Centroid_3T = vec(readdlm(PathData * "Torquato_NR_N$(NSides)_Alfa0P0_Area3_SigmaCuadrada.csv"));
σ2_Centroid_4T = vec(readdlm(PathData * "Torquato_NR_N$(NSides)_Alfa0P0_Area4_SigmaCuadrada.csv"));

##### Visualización de la gráfica de fluctuaciones (Hiperuniformidad)

In [ ]:
#Creación del lienzo en blanco donde se dibujarán las gráficas
Sigma2 = Figure(size = (1000, 500), figure_padding = 15);

#Definición de los ejes X-Y donde se construirá una gráfica
ax = Axis(
          Sigma2[1, 1],
          xlabel = L"R",
          ylabel = L"\sigma^{2}(R)/R",
          limits = ((0, 1000), nothing),                       #Límites de los ejes X y Y
          xticklabelsize = 33, yticklabelsize = 33,            #Tamaño de la fuente usada para los números en los ejes
          xlabelsize = 35, ylabelsize = 35,                    #Tamaño de la fuente usada para las etiquetas de los ejes
          xgridvisible = false, ygridvisible = false,          #Quitar el mallado de fondo
          topspinevisible = false, rightspinevisible = false   #Quitamos los lados superior y derecho del box donde se grafica
         );

#Definición de la paleta de colores a usar
raw_colors = [
              [58, 12, 163]/255,  #Color de la primera tesela
              [247, 23, 53]/255,  #Color de la segunda tesela
              [32, 163, 158]/255, #Color de la tercera tesela
              [255, 186, 73]/255  #Color de la cuarta tesela
             ];
mis_colores = [RGB(c[1], c[2], c[3]) for c in raw_colors]; #Creamos un vector de colores tipo RGB

#Generamos las curvas de las fluctuaciones en el número de sitios (sigma^2(R))
lines!(
       ax, R, (σ2_Centroid_1T ./ R) .+ 0.85, 
       color = mis_colores[1],
       label = L"AT = 1",
       linewidth = 1
      );

lines!(
       ax, R, (σ2_Centroid_2T ./ R) .+ 0.48, 
       color = mis_colores[2],
       label = L"AT = 2",
       linewidth = 1
      );

lines!(
       ax, R, (σ2_Centroid_3T ./ R) .+ 0.20, 
       color = mis_colores[3],
       label = L"AT = 3",
       linewidth = 1
      );

lines!(
       ax, R, σ2_Centroid_4T ./ R, 
       color = mis_colores[4],
       label = L"AT = 4",
       linewidth = 1
      );

#Dibujamos las líneas verticales que corresponden a los múltiplos de la escala de longitud
x_verticales = [j * AproxLambda(NSides) for j in 1:11];

vlines!(
        ax, x_verticales, 
        ymin = 0, ymax = 1.8,
        color = (:black, 0.75),
        linestyle = :dash
       );

#Añadimos los diamantes correspondientes a la longitud de escala kappa
Rho = 1.2712590235307708;       #Densidad de sitios en una vecindad cuasiperiódica (Decoración vértices)
FactNorm = 2*sqrt(π*Rho);       #Factor de normalización para las longitudes, genera densidades constantes
κ_23 = (NSides / 4) * FactNorm; #Valor de la escala de longitud Kappa
Index_Kappa = findfirst(x -> x >= κ_23, R);

# --- Tesela más pequeña ---
scatter!(
         ax, [κ_23], [(σ2_Centroid_1T[Index_Kappa] ./ R[Index_Kappa]) + 0.85],
         color = :red,
         marker = :diamond,
         alpha = 0.6
        );

# --- Tesela segunda más pequeña ---
scatter!(
         ax, [κ_23], [(σ2_Centroid_2T[Index_Kappa] ./ R[Index_Kappa]) + 0.48],
         color = :red,
         marker = :diamond,
         alpha = 0.6
        );

# --- Tesela tercera más pequeña ---
scatter!(
         ax, [κ_23], [(σ2_Centroid_3T[Index_Kappa] ./ R[Index_Kappa]) + 0.20],
         color = :red,
         marker = :diamond,
         alpha = 0.6
        );

# --- Tesela cuarta más pequeña ---
scatter!(
         ax, [κ_23], [(σ2_Centroid_4T[Index_Kappa] ./ R[Index_Kappa])],
         color = :red,
         marker = :diamond,
         alpha = 0.6
        );

Sigma2

##### Generación de las vecindades cuadradas con el decorado correspondiente (1st, 2nd, 3rd, ... Tile)

In [ ]:
NSides = 23;        #Simetría rotacional del sistema cuasiperiódico
Areas_Values = [0.136167, 0.269797, 0.398401, 0.519584, 0.631088, 0.730836, 0.81697, 0.887885, 0.942261, 0.979084, 0.997669]; #Arreglo con las áreas, de menor a mayor, de las posibles teselas
Error_Margin = 12;  #Margen de error de los enteros asociados a los vectores estrella
SLCluster = 40;     #Semilado de la vecindad cuasiperiódica cuadrada a generar

Star_Vectors = [[BigFloat(1),0]]; #Conjunto de vectores estrella
for i in 1:(NSides-1)
    push!(Star_Vectors, [cos((2*i)*pi/NSides), sin((2*i)*pi/NSides)]); #Vértices de un polígono unitario regular de NSides
end
Alphas_Array = fill(0.0, NSides); #Arreglo con los valores de los parámetros alpha
Average_Distance_Stripes = fill(NSides/2, NSides); #Arreglo con la separación promedio entre franjas cuasiperiódicas

#Generamos las teselas de la vecindad local del sist. cuasiperiódico
Dual_Points = region_Local_Voronoi(Error_Margin, Average_Distance_Stripes, Star_Vectors, Alphas_Array, APoint);
TPAG = [[] for i in 1:length(Areas_Values)]; #Tiles Per Area Group

for i in 1:4:(length(Dual_Points)-3)
    #Coordenadas X, Y de los sitios de la retícula cuasiperiódica agrupados en teselas
    XX = [Dual_Points[i][1], Dual_Points[i+1][1], Dual_Points[i+2][1], Dual_Points[i+3][1], Dual_Points[i][1]];
    YY = [Dual_Points[i][2], Dual_Points[i+1][2], Dual_Points[i+2][2], Dual_Points[i+3][2], Dual_Points[i][2]];

    Tesela = [(XX[j], YY[j]) for j in 1:length(XX)]; #Tesela del sistema cuasiperiódico
        
    #Calculamos el centroide de cada tesela
    Centroid = [(Dual_Points[i][1] + Dual_Points[i+1][1] + Dual_Points[i+2][1] + Dual_Points[i+3][1])/4,
                (Dual_Points[i][2] + Dual_Points[i+1][2] + Dual_Points[i+2][2] + Dual_Points[i+3][2])/4];
            
    #Guardamos únicamente la información de las teselas que caen dentro del clúster cuadrado
    if (-SLCluster < (Centroid[1] - APoint[1]) < SLCluster) && (-SLCluster < (Centroid[2] - APoint[2]) < SLCluster)
        A = round(Float64(area(Tesela)), digits = 6); #Area de la tesela
        Index = findfirst(x -> x == A, Areas_Values); #Índice del área de la tesela en el arreglo de áreas
        push!(TPAG[Index], [[e[1], e[2]] - APoint for e in Tesela]); #Guardamos tesela de cluster principal
    end
end

Dual_Points = nothing; #Liberamos memoria de la vecindad local del sistema cuasiperiódico

##### Graficamos la vecindad de las teselas más pequeñas

In [ ]:
AreaIndex = 1;                            #Índice del área de las teselas que buscamos graficar
Tiles_Array = TPAG[AreaIndex];            #Arreglo con las diferentes teselas de menor área
ColorTesela = mis_colores[AreaIndex];     #Arreglo con las coordenadas RGB del color de la tesela

#Crear Figura y Eje
Sample_T1 = Figure(size = (500, 500), figure_padding = 0);
ax = Axis(Sample_T1[1, 1], aspect = DataAspect());
hidedecorations!(ax); #Quita ticks, etiquetas y grid
hidespines!(ax);      #Quita el recuadro negro del borde

#Convertir datos y Graficar
lista_poligonos = [Point2f[(Tesela[i][1], Tesela[i][2]) for i in 1:5] for Tesela in Tiles_Array];

poly!(
      ax, lista_poligonos,
      color = ColorTesela,
      strokecolor = ColorTesela,
      strokewidth = 0.2
     );

Rho = 1.2712590235307708;           #Densidad de sitios en una vecindad cuasiperiódica (Decoración vértices)
FactNorm = 2*sqrt(π*Rho);           #Factor de normalización para las longitudes, genera densidades constantes
FactNorm = 1;
κ_23 = (NSides / 4) * FactNorm;     #Valor de la escala de longitud Kappa

#Definimos un círculo: Circle(centro, radio)
circulo = Circle(Point2f(0, 0), κ_23/2);
scatter!(ax, [0], [0], color = :red, markersize = 2);
poly!(ax, circulo, color = :transparent, strokecolor = :red, strokewidth = 1);

Sample_T1

In [ ]:
AreaIndex = 2;                            #Índice del área de las teselas que buscamos graficar
Tiles_Array = TPAG[AreaIndex];            #Arreglo con las diferentes teselas de menor área
ColorTesela = mis_colores[AreaIndex];     #Arreglo con las coordenadas RGB del color de la tesela

#Crear Figura y Eje
Sample_T2 = Figure(size = (500, 500), figure_padding = 0);
ax = Axis(Sample_T2[1, 1], aspect = DataAspect());
hidedecorations!(ax); #Quita ticks, etiquetas y grid
hidespines!(ax);      #Quita el recuadro negro del borde

#Convertir datos y Graficar
lista_poligonos = [Point2f[(Tesela[i][1], Tesela[i][2]) for i in 1:5] for Tesela in Tiles_Array];

poly!(
      ax, lista_poligonos,
      color = ColorTesela,
      strokecolor = ColorTesela,
      strokewidth = 0.2
     );

Sample_T2

In [ ]:
AreaIndex = 3;                            #Índice del área de las teselas que buscamos graficar
Tiles_Array = TPAG[AreaIndex];            #Arreglo con las diferentes teselas de menor área
ColorTesela = mis_colores[AreaIndex];     #Arreglo con las coordenadas RGB del color de la tesela

#Crear Figura y Eje
Sample_T3 = Figure(size = (500, 500), figure_padding = 0);
ax = Axis(Sample_T3[1, 1], aspect = DataAspect());
hidedecorations!(ax); #Quita ticks, etiquetas y grid
hidespines!(ax);      #Quita el recuadro negro del borde

#Convertir datos y Graficar
lista_poligonos = [Point2f[(Tesela[i][1], Tesela[i][2]) for i in 1:5] for Tesela in Tiles_Array];

poly!(
      ax, lista_poligonos,
      color = ColorTesela,
      strokecolor = ColorTesela,
      strokewidth = 0.2
     );

Sample_T3

In [ ]:
AreaIndex = 4;                            #Índice del área de las teselas que buscamos graficar
Tiles_Array = TPAG[AreaIndex];            #Arreglo con las diferentes teselas de menor área
ColorTesela = mis_colores[AreaIndex];     #Arreglo con las coordenadas RGB del color de la tesela

#Crear Figura y Eje
Sample_T4 = Figure(size = (500, 500), figure_padding = 0);
ax = Axis(Sample_T4[1, 1], aspect = DataAspect());
hidedecorations!(ax); #Quita ticks, etiquetas y grid
hidespines!(ax);      #Quita el recuadro negro del borde

#Convertir datos y Graficar
lista_poligonos = [Point2f[(Tesela[i][1], Tesela[i][2]) for i in 1:5] for Tesela in Tiles_Array];

poly!(
      ax, lista_poligonos,
      color = ColorTesela,
      strokecolor = ColorTesela,
      strokewidth = 0.2
     );

Sample_T4

In [ ]:
# --- Definición de la paleta de colores a emplear ---
raw_colors = [
              [58, 12, 163]/255, 
              [247, 23, 53]/255, 
              [32, 163, 158]/255, 
              [255, 186, 73]/255
             ];
mis_colores = [RGB(c[1], c[2], c[3]) for c in raw_colors]; #Creamos un vector de colores tipo RGB

# --- Generamos el lienzo en blanco donde se dibujarán las gráficas ---
fig = Figure(size = (1200, 500));

# --- Generamos el eje donde se dibujará la gráfica de hiperuniformidad ---
Sigma2 = Axis(
              fig[1, 2],
              xlabel = L"R",
              ylabel = L"\sigma^{2}(R)/R",
              limits = ((0, 700), nothing),                       #Límites de los ejes X y Y
              xticks = [0, 250, 500],                             #Posiciones donde colocar los ticks en el eje X
              yticks = (
                        [0, 1.5],                                 #Posiciones donde colocar los ticks en el eje Y
                        [rich("0.0"), rich("1.5")]                #Etiquetas de los ticks en el eje Y
                       ),
              xticklabelsize = 35, yticklabelsize = 35,
              xlabelsize = 35, ylabelsize = 35,
              xgridvisible = false, ygridvisible = false,         #Quitar el mallado de fondo
              topspinevisible = false, rightspinevisible = false  #Quitamos los lados superior y derecho del box donde se grafica
             );

#Generamos las gráficas de las varianzas por subgrupo de teselas
lines!(
       Sigma2, R, (σ2_Centroid_1T ./ R) .+ 0.85, 
       color = mis_colores[1],
       label = L"AT = 1",
       linewidth = 1
      );
lines!(
       Sigma2, R, (σ2_Centroid_2T ./ R) .+ 0.48, 
       color = mis_colores[2],
       label = L"AT = 2",
       linewidth = 1
      );
lines!(
       Sigma2, R, (σ2_Centroid_3T ./ R) .+ 0.20, 
       color = mis_colores[3],
       label = L"AT = 3",
       linewidth = 1
      );
lines!(
       Sigma2, R, σ2_Centroid_4T ./ R, 
       color = mis_colores[4],
       label = L"AT = 4",
       linewidth = 1
      );

#Líneas verticales (Múltiplos de Length Scale)
x_verticales = [j * AproxLambda(NSides) for j in 1:11]
vlines!(
        Sigma2, x_verticales, 
        ymin = 0, ymax = 1.8, #vlines usa coordenadas 0 a 1 relativas al eje, o absolutas si no se especifica
        color = (:black, 0.75),   #Color y alfa en una tupla
        linestyle = :dash
       );

#Generamos los diamantes que indican a la escala kappa
#Añadimos los diamantes correspondientes a la longitud de escala kappa
Rho = 1.2712590235307708; #Densidad de sitios en una vecindad cuasiperiódica (Decoración vértices)
FactNorm = 2*sqrt(π*Rho); #Factor de normalización para las longitudes, genera densidades constantes
κ_23 = (NSides / 4) * FactNorm; #Valor de la escala de longitud Kappa
Index_Kappa = findfirst(x -> x >= κ_23, R);

scatter!(
         Sigma2, [κ_23], [(σ2_Centroid_1T[Index_Kappa] ./ R[Index_Kappa]) + 0.85],
         color = :red,
         marker = :diamond,
         alpha = 0.6
        );
scatter!(
         Sigma2, [κ_23], [(σ2_Centroid_2T[Index_Kappa] ./ R[Index_Kappa]) + 0.48],
         color = :red,
         marker = :diamond,
         alpha = 0.6
        );
scatter!(
         Sigma2, [κ_23], [(σ2_Centroid_3T[Index_Kappa] ./ R[Index_Kappa]) + 0.20],
         color = :red,
         marker = :diamond,
         alpha = 0.6
        );
scatter!(
         Sigma2, [κ_23], [(σ2_Centroid_4T[Index_Kappa] ./ R[Index_Kappa])],
         color = :red,
         marker = :diamond,
         alpha = 0.6
        );

# --- Generamos el mallado donde vamos a colocar las cuatro sub gráficas con la muestra del teselado por subgrupo de teselas ---
gl_derecha = GridLayout(fig[1, 1]);

#Fila 1, Columna 1
Sample_T1 = Axis(gl_derecha[1, 1], aspect = DataAspect());
hidedecorations!(Sample_T1); #Quita ticks, etiquetas y grid
hidespines!(Sample_T1);      #Quita el recuadro negro del borde

AreaIndex = 1;                            #Índice del área de las teselas que buscamos graficar
Tiles_Array = TPAG[AreaIndex];            #Arreglo con las diferentes teselas de menor área
ColorTesela = mis_colores[AreaIndex];     #Arreglo con las coordenadas RGB del color de la tesela

lista_poligonos = [Point2f[(Tesela[i][1], Tesela[i][2]) for i in 1:5] for Tesela in Tiles_Array];
poly!(
      Sample_T1, lista_poligonos,
      color = ColorTesela,
      strokecolor = ColorTesela,
      strokewidth = 0.2
     );

#Dibujamos un círculo de referencia para conocer la escala kappa en nuestro espacio físico
Rho = 1.2712590235307708;       #Densidad de sitios en una vecindad cuasiperiódica (Decoración vértices)
FactNorm = 2*sqrt(π*Rho);       #Factor de normalización para las longitudes, genera densidades constantes
FactNorm = 1;                   #Valor por default para obviar la presencia de un factor de normalización (en el espacio real no aplicamos el factor)
κ_23 = (NSides / 4) * FactNorm; #Valor de la escala de longitud Kappa

Circulo = Circle(Point2f(0, 0), κ_23); #Definir un círculo: Circle(centro, radio)
scatter!(Sample_T1, [0], [0], color = :red, markersize = 0.25);
poly!(Sample_T1, Circulo, color = :transparent, strokecolor = :red, strokewidth = 1);

#Fila 1, Columna 2
Sample_T2 = Axis(gl_derecha[1, 2], aspect = DataAspect());
hidedecorations!(Sample_T2); #Quita ticks, etiquetas y grid
hidespines!(Sample_T2);      #Quita el recuadro negro del borde

AreaIndex = 2;                            #Índice del área de las teselas que buscamos graficar
Tiles_Array = TPAG[AreaIndex];            #Arreglo con las diferentes teselas de menor área
ColorTesela = mis_colores[AreaIndex];     #Arreglo con las coordenadas RGB del color de la tesela

lista_poligonos = [Point2f[(Tesela[i][1], Tesela[i][2]) for i in 1:5] for Tesela in Tiles_Array];
poly!(
      Sample_T2, lista_poligonos,
      color = ColorTesela,
      strokecolor = ColorTesela,
      strokewidth = 0.2
     );

#Fila 2, Columna 1
Sample_T3 = Axis(gl_derecha[2, 1], aspect = DataAspect());
hidedecorations!(Sample_T3); #Quita ticks, etiquetas y grid
hidespines!(Sample_T3);      #Quita el recuadro negro del borde

AreaIndex = 3;                            #Índice del área de las teselas que buscamos graficar
Tiles_Array = TPAG[AreaIndex];            #Arreglo con las diferentes teselas de menor área
ColorTesela = mis_colores[AreaIndex];     #Arreglo con las coordenadas RGB del color de la tesela

lista_poligonos = [Point2f[(Tesela[i][1], Tesela[i][2]) for i in 1:5] for Tesela in Tiles_Array];
poly!(
      Sample_T3, lista_poligonos,
      color = ColorTesela,
      strokecolor = ColorTesela,
      strokewidth = 0.2
     );

#Fila 2, Columna 2
Sample_T4 = Axis(gl_derecha[2, 2], aspect = DataAspect());
hidedecorations!(Sample_T4); #Quita ticks, etiquetas y grid
hidespines!(Sample_T4);      #Quita el recuadro negro del borde

AreaIndex = 4;                            #Índice del área de las teselas que buscamos graficar
Tiles_Array = TPAG[AreaIndex];            #Arreglo con las diferentes teselas de menor área
ColorTesela = mis_colores[AreaIndex];     #Arreglo con las coordenadas RGB del color de la tesela

lista_poligonos = [Point2f[(Tesela[i][1], Tesela[i][2]) for i in 1:5] for Tesela in Tiles_Array];
poly!(
      Sample_T4, lista_poligonos,
      color = ColorTesela,
      strokecolor = ColorTesela,
      strokewidth = 0.2
     );

#Ajustar el layout para eliminar espacios blancos innecesarios
colgap!(gl_derecha, 0);                   #Separación entre las columnas del grid 2x2
rowgap!(gl_derecha, 0);                   #Separación entre las filas del grid 2x2
colsize!(fig.layout, 1, Relative(0.38));  #Tamaño relativo de la gráfica de la izquierda con respecto a la gráfica de la derecha

fig